# 3. Baseline Models

## Purpose
Establish a performance benchmark by training and cross validating multiple
models on the preprocessed data. All decisions about which model to take
forward are made using cross validation scores only.

## Inputs
- `data/processed/` — preprocessed train and test features and labels
- `data/raw/` — raw data reloaded for CatBoost
- `config.yaml` — baseline model settings
- `artifacts/preprocessor.pkl` — fitted preprocessing pipeline

## Outputs
- `models/baseline/` — all trained baseline models
- `artifacts/baseline_trainer.pkl` — trainer state including leaderboard
  and CV results for all models

## Decisions for the user
- Set `baseline.run_all` in `config.yaml` — true runs all models for the
  detected task, false runs only the models listed under `baseline.models`
- Set `baseline.cv_folds` in `config.yaml` — typically 5 for most projects
- Review the leaderboard and select which models to take forward to tuning
  via `tuning.models_to_tune` in `config.yaml`

## Important note
The preprocessor passed to the trainer is unfitted. It is fitted inside
each CV fold on training data only, preventing data leakage into the
validation fold.

In [1]:
import pandas as pd
import numpy as np
import joblib
import copy

from src.config.settings import load_config
from src.config.paths import (
    PROCESSED_DATA_DIR,
    BASELINE_MODELS_DIR,
    ARTIFACTS_DIR
)
from src.features.pipeline import FeatureEngineeringPipeline
from src.models.training.trainer import ModelTrainer
from src.models.training.model_registry import MODEL_REGISTRY
from src.models.training.save_load import save_model

## 1. Load config

In [2]:
config = load_config()

TARGET = config["target"]
BASELINE_CONFIG = config["baseline"]
PIPELINE_CONFIG = config["pipeline"]

print(f"Target:         {TARGET}")
print(f"Run all models: {BASELINE_CONFIG['run_all']}")
print(f"CV folds:       {BASELINE_CONFIG['cv_folds']}")

Target:         Survived
Run all models: True
CV folds:       5


## 2. Load processed data

In [3]:
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (712, 106)
X_test shape:  (179, 106)


## 3. Load raw data for CatBoost
The same split parameters from config are used to ensure
identical train/test splits across all models.

In [4]:
from src.data.load_data import load_raw_data
from sklearn.model_selection import train_test_split

DROP_COLS = config["drop_columns"]
SPLIT_CONFIG = config["split"]

df = load_raw_data()
X_raw = df.drop(columns=[TARGET] + DROP_COLS)
y_all = df[TARGET]

stratify_col = y_all if SPLIT_CONFIG["stratify"] else None

X_train_raw, X_test_raw, _, _ = train_test_split(
    X_raw, y_all,
    test_size=SPLIT_CONFIG["test_size"],
    random_state=SPLIT_CONFIG["random_state"],
    stratify=stratify_col
)

print(f"X_train_raw shape: {X_train_raw.shape}")
print(f"X_test_raw shape:  {X_test_raw.shape}")

X_train_raw shape: (712, 10)
X_test_raw shape:  (179, 10)


## 4. Initialise preprocessor
The preprocessor must be unfitted at this stage. The trainer will deep copy
and fit it inside each CV fold to prevent data leakage.

In [5]:
preprocessor = FeatureEngineeringPipeline(config=PIPELINE_CONFIG)

## 5. Initialise trainer

In [6]:
trainer = ModelTrainer(
    model_registry=MODEL_REGISTRY,
    random_state=BASELINE_CONFIG["random_state"]
)

## 6. Train baseline models
Trains all models appropriate for the detected task type using stratified
k-fold cross validation. Metrics are averaged across folds.

In [7]:
trainer.fit(
    X=X_train_raw,
    y=y_train.values,
    preprocessor=preprocessor,
    cv=BASELINE_CONFIG["cv_folds"],
    run_all=BASELINE_CONFIG["run_all"],
    model_subset=BASELINE_CONFIG.get("models", None)
)

Task detected: classification

Training: logistic_regression


C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project


Training: random_forest_classifier


C:\ML\ML Workflow\Projects\Project Name Template\src\features\pipeline.py:188: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}__hash_{j}"] = hashed[:, j]
C:\ML\ML Workflow\Projects\Project Name Template\src\features\pipeline.py:188: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}__hash_{j}"] = hashed[:, j]
C:\ML\ML Workflow\Projects\Project Name Template\src\features\pipeline.py:188: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has p


Training: extra_trees_classifier


...

[LightGBM] [Info] Number of positive: 219, number of negative: 351
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000888 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 244
[LightGBM] [Info] Number of data points in the train set: 570, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.384211 -> initscore=-0.471714
[LightGBM] [Info] Start training from score -0.471714
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project

[LightGBM] [Info] Number of positive: 219, number of negative: 351
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000796 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 242
[LightGBM] [Info] Number of data points in the train set: 570, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.384211 -> initscore=-0.471714
[LightGBM] [Info] Start training from score -0.471714
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000333 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 302
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 54
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

## 7. Leaderboard
Review CV scores for all models. Use this to decide which models to take
forward to hyperparameter tuning — update `tuning.models_to_tune` in
`config.yaml` accordingly.

In [8]:
trainer.leaderboard_

,model_name,primary_metric_name,primary_metric_value,roc_auc,f1,precision,recall,accuracy,log_loss,rmse,mae,r2,training_time_sec
0,catboost_classifier,roc_auc,0.892414,0.892414,0.819671,0.824671,0.818773,0.830099,0.395765,None,None,None,147.134266
1,xgboost_classifier,roc_auc,0.882887,0.882887,0.801587,0.806970,0.799493,0.813277,0.429313,None,None,None,2.447195
2,lightgbm_classifier,roc_auc,0.878047,0.878047,0.798260,0.802615,0.796507,0.810430,0.499591,None,None,None,10.681830
3,random_forest_classifier,roc_auc,0.875515,0.875515,0.802568,0.814357,0.797360,0.817483,0.425662,None,None,None,5.265660
4,logistic_regression,roc_auc,0.855087,0.855087,0.781144,0.789263,0.778258,0.796366,0.458473,None,None,None,1.438712
5,extra_trees_classifier,roc_auc,0.852505,0.852505,0.776541,0.784853,0.773656,0.790781,0.564560,None,None,None,4.684412


## 8. Identify best model
The best model by CV score is identified here for reference. This is the
benchmark to beat in the feature engineering and tuning notebooks.

In [9]:
best_model_name = trainer.leaderboard_.iloc[0]["model_name"]
best_model = trainer.models_[best_model_name]

print(f"Best model from CV: {best_model_name}")
print(f"CV {trainer.leaderboard_.iloc[0]['primary_metric_name']}: "
      f"{trainer.leaderboard_.iloc[0]['primary_metric_value']:.4f}")

Best model from CV: catboost_classifier
CV roc_auc: 0.8924


## 9. Save baseline models and trainer state
All models are saved individually to `models/baseline/`. The trainer state
is saved to `artifacts/` so downstream notebooks can access the leaderboard
and CV results for comparison.

In [10]:
BASELINE_MODELS_DIR.mkdir(parents=True, exist_ok=True)

for model_name, model in trainer.models_.items():
    save_model(model, BASELINE_MODELS_DIR / f"{model_name}.pkl")

print(f"All baseline models saved to {BASELINE_MODELS_DIR}")

# %%
# 10. Save Trainer State
joblib.dump(trainer, ARTIFACTS_DIR / "baseline_trainer.pkl")
print(f"Trainer saved to {ARTIFACTS_DIR / 'baseline_trainer.pkl'}")

Saving model: C:\ML\ML Workflow\Projects\Project Name Template\models\baseline\logistic_regression.pkl

Saving model: C:\ML\ML Workflow\Projects\Project Name Template\models\baseline\random_forest_classifier.pkl

Saving model: C:\ML\ML Workflow\Projects\Project Name Template\models\baseline\extra_trees_classifier.pkl

Saving model: C:\ML\ML Workflow\Projects\Project Name Template\models\baseline\xgboost_classifier.pkl

Saving model: C:\ML\ML Workflow\Projects\Project Name Template\models\baseline\lightgbm_classifier.pkl

Saving model: C:\ML\ML Workflow\Projects\Project Name Template\models\baseline\catboost_classifier.pkl

All baseline models saved to C:\ML\ML Workflow\Projects\Project Name Template\models\baseline
Trainer saved to C:\ML\ML Workflow\Projects\Project Name Template\artifacts\baseline_trainer.pkl
